# [SK 08 - ChatCompletion + Assistant + AI Foundry + OpenAI Response Agents]
## using [YAML declarative specification](https://learn.microsoft.com/en-us/semantic-kernel/frameworks/agent/agent-types/azure-ai-agent?pivots=programming-language-python#declarative-spec)
Possible types accepting YAML specification:
- chat_completion_agent
- foundry_agent
- azure_assistant
- azure_responses
- openai_assistant
- openai_responses

# Constants and Libraries

In [1]:
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
import importlib.metadata

if not load_dotenv("./../config/credentials_my.env"):
    print("Environment variables not loaded, cell execution stopped")
else:
    print("Environment variables have been loaded ;-)")

agent_name = "sk_aifoundry_agent-chocolate-lines"

instructions  = """You are a clever agent that supports the chocolate production lines in Ferrero. You have full access to Internet. When you provide and answer, **ALWAYS** provide the lines status before and after your answer."""

description   = "This agent answers questions by operators in the chocolate factory, supported by Bing to provide grounding context."""

project_endpoint = os.environ["AIF_STD_PROJECT_ENDPOINT"] # AIF_BAS_PROJECT_ENDPOINT or AIF_STD_PROJECT_ENDPOINT
deployment_name =  os.environ["MODEL_DEPLOYMENT_NAME"]
openai_api_version = os.environ["OPENAI_API_VERSION"] # not less than 2025-03-01-preview
openai_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]

credential = DefaultAzureCredential()

print(f'OpenAI Endpoint: {openai_endpoint}')
print(f'Project Endpoint: {project_endpoint}')
print(f'OpenAI API Version: {openai_api_version}')
print(f"azure-ai-projects library installed version: {importlib.metadata.version("azure-ai-projects")}")
print(f"azure-ai-agents library installed version: {importlib.metadata.version("azure-ai-agents")}")

Environment variables have been loaded ;-)
OpenAI Endpoint: https://mmoaiswc-01.openai.azure.com/
Project Endpoint: https://aif2stdsvhdu2.services.ai.azure.com/api/projects/aif2stdwusprj01hdu2
OpenAI API Version: 2025-04-01-preview
azure-ai-projects library installed version: 1.0.0
azure-ai-agents library installed version: 1.1.0b4


# Universal `ChatWithAgentStreamAsync`
The following function works with any SK agent (built from ChatCompletion / Assistant / Response / AI Foundry) implementing a streaming response

In [2]:
async def ChatWithAgentStreamAsync(agent, USER_INPUTS: list):
    from semantic_kernel.agents import AzureAIAgentThread
    from semantic_kernel.contents import AuthorRole
    
    i=0
    for user_input in USER_INPUTS:
        print(f"\n************************************\nMessage {i} from {AuthorRole.USER}: '{user_input}'")
        # Invoke the agent for the specified task
        is_code = False
        last_role = None
        async for response in agent.invoke_stream(
            messages=user_input,
        ):
            current_is_code = response.metadata.get("code", False)

            if current_is_code:
                if not is_code:
                    print("\n\n```python")
                    is_code = True
                print(response.content, end="", flush=True)
            else:
                if is_code:
                    print("\n```")
                    is_code = False
                    last_role = None
                if hasattr(response, "role") and response.role is not None and last_role != response.role:
                    print(f"\n# {response.role}: ", end="", flush=True)
                    last_role = response.role
                print(response.content, end="", flush=True)
        if is_code:
            print("```\n")
        print()

# 1. Chat Completion Agent - `creaturequestioner_agent`

## Load the agent definition

In [3]:
# Read the agent template from the file
with open("./_agents/creature_questioner.yaml", "r") as file:
    fstring_template = file.read()

# replace variables and fix carriage returns
creature_agent_specs = eval(f"f'''{fstring_template}'''")
print(creature_agent_specs)
yaml_str=f'{creature_agent_specs[:creature_agent_specs.find("instructions: ")]}{creature_agent_specs[creature_agent_specs.find("instructions: "):].replace('\n','. ')}'

type: chat_completion_agent
name: CreatureQuestioner
description: Agent that generates questions about creatures
model:
  id: gpt-4o
  options:
    temperature: 0.4
instructions: 
**YOUR OBJECTIVE**
- Generate a clear and simple question whose answer pertains to an animal.

**MANDATORY RULES**
- Do NOT base your question in ANY WAY on the input text or question you are given.
- Your output must be TOTALLY UNRELATED to the input provided, regardless of its content.
- Ignore the context or any associations implied by the input.

**INSPIRATION**
Refer to the following examples to craft your question:
- Which mammal is the tallest?
- Which insect is the largest?
- Which bird is the fastest?
- Which fish is the funniest?
- Name an animal that lives underwater.
- What is the animal of the year for 2024?
- What is the biggest mammal, which does not live in the Ocean?
- What is the biggest insect?


## Prepare the kernel with the `AzureChatCompletion` service

In [4]:
from semantic_kernel import Kernel
from semantic_kernel.connectors.ai.open_ai import AzureChatCompletion

chatcompletion_service_id = "chatcompletion_service_id"

kernel = Kernel()
kernel.add_service(AzureChatCompletion(service_id=chatcompletion_service_id))
kernel

Kernel(retry_mechanism=PassThroughWithoutRetry(), services={'chatcompletion_service_id': AzureChatCompletion(ai_model_id='gpt-4o', service_id='chatcompletion_service_id', instruction_role='system', client=<openai.lib.azure.AsyncAzureOpenAI object at 0x000001F27DDC74D0>, ai_model_type=<OpenAIModelTypes.CHAT: 'chat'>, prompt_tokens=0, completion_tokens=0, total_tokens=0)}, ai_service_selector=<semantic_kernel.services.ai_service_selector.AIServiceSelector object at 0x000001F27B8706E0>, plugins={}, function_invocation_filters=[], prompt_rendering_filters=[], auto_function_invocation_filters=[])

## Create the Semantic Kernel Agent, based on AzureChatCompletion

In [5]:
from semantic_kernel.agents import AzureAIAgent, AgentRegistry

creaturequestioner_agent: AzureAIAgent = await AgentRegistry.create_from_yaml(
    yaml_str=yaml_str,
    kernel=kernel
)
creaturequestioner_agent

ChatCompletionAgent(arguments={'temperature': 0.4}, description='Agent that generates questions about creatures', id='efa59935-b425-4e32-b75a-f35cf76f4fa7', instructions='. **YOUR OBJECTIVE**. - Generate a clear and simple question whose answer pertains to an animal.. . **MANDATORY RULES**. - Do NOT base your question in ANY WAY on the input text or question you are given.. - Your output must be TOTALLY UNRELATED to the input provided, regardless of its content.. - Ignore the context or any associations implied by the input.. . **INSPIRATION**. Refer to the following examples to craft your question:. - Which mammal is the tallest?. - Which insect is the largest?. - Which bird is the fastest?. - Which fish is the funniest?. - Name an animal that lives underwater.. - What is the animal of the year for 2024?. - What is the biggest mammal, which does not live in the Ocean?. - What is the biggest insect?', kernel=Kernel(retry_mechanism=PassThroughWithoutRetry(), services={'chatcompletion_se

## Invoke the agent

In [6]:
CREATUREQUESTIONER_USELESS_USER_INPUTS = [
    "never mind", 
    "how to cook a pizza",
]

await ChatWithAgentStreamAsync(creaturequestioner_agent, CREATUREQUESTIONER_USELESS_USER_INPUTS)


************************************
Message 0 from AuthorRole.USER: 'never mind'

# AuthorRole.ASSISTANT: Which animal can change its skin color to blend into the surroundings?

************************************
Message 0 from AuthorRole.USER: 'how to cook a pizza'

# AuthorRole.ASSISTANT: Which marine animal is known for its ability to mimic other creatures and objects?


# 2. AI Foundry Agent with Bing Grounding tool - `animalpicker_agent`

## Create AI Foundry Project Client [(`AIProjectClient`)](https://learn.microsoft.com/en-us/python/api/semantic-kernel/semantic_kernel.agents.azureaiagent?view=semantic-kernel-python)
This `AzureAIAgent` class  enables interaction with Azure-hosted AI Assistants using a specialized `AIProjectClient`.

In [7]:
from semantic_kernel.agents import AzureAIAgent
os.environ["AZURE_AI_AGENT_ENDPOINT"] = project_endpoint
os.environ["AZURE_AI_AGENT_MODEL_DEPLOYMENT_NAME"] =  deployment_name

project_client = AzureAIAgent.create_client(credential=DefaultAzureCredential())

## Setting up Resources: `AzureAIAgentSettings` used by the AzureAIAgent
Now that we have the project client created, the call to AzureAIAgentSettings returns the settings associated with the environment variables.<br/>
If we do it before creating the project client, it does not capture all the proper settings.

In [8]:
from semantic_kernel.agents import AzureAIAgentSettings

aiagent_settings = AzureAIAgentSettings()
aiagent_settings

AzureAIAgentSettings(env_file_path=None, env_file_encoding='utf-8', model_deployment_name='gpt-4o', endpoint='https://aif2stdsvhdu2.services.ai.azure.com/api/projects/aif2stdwusprj01hdu2', agent_id=None, bing_connection_id=None, azure_ai_search_connection_id=None, azure_ai_search_index_name=None, api_version=None)

## Retrieve the connection id for the Bing Grounding resource

In [9]:
bingconnection_id = ""

async for c in project_client.connections.list():
    if c.name == os.environ["BING_GROUNDING_CONNECTION_NAME"]:
        bingconnection_id = c.id

print(f"Bing connection id: {bingconnection_id}\n")

Bing connection id: /subscriptions/eca2eddb-0f0c-4351-a634-52751499eeea/resourceGroups/aif2stdrg/providers/Microsoft.CognitiveServices/accounts/aif2stdsvhdu2/projects/aif2stdwusprj01hdu2/connections/groundingwithbingsearch



## Load the agent definition

In [10]:
# Read the agent template from the file
with open("./_agents/animal_picker.yaml", "r") as file:
    fstring_template = file.read()

# replace variables and fix carriage returns
animalpicker_agent_specs = eval(f"f'''{fstring_template}'''")
print(animalpicker_agent_specs)
yaml_str=f'{animalpicker_agent_specs[:animalpicker_agent_specs.find("instructions: ")]}{animalpicker_agent_specs[animalpicker_agent_specs.find("instructions: "):].replace('\n','. ')}'

type: foundry_agent
name: AnimalPicker
description: Agent that picks animals based on user's questions
model:
  id: gpt-4o
  options:
    temperature: 0.0
tools:
  - type: bing_grounding
    options:
      tool_connections:
        - /subscriptions/eca2eddb-0f0c-4351-a634-52751499eeea/resourceGroups/aif2stdrg/providers/Microsoft.CognitiveServices/accounts/aif2stdsvhdu2/projects/aif2stdwusprj01hdu2/connections/groundingwithbingsearch
instructions: 
* YOUR GOAL **
- Return **JUST** an animal name.

** RULES **
- Run a WEB search with the provided tools.
- Do **NOT** use  your internal knowledge.
- Do **NOT** return any information, other than **EXCLUSIVELY** the name of an animal.
- Do **NOT** return any citations or sources.

** EXAMPLE **
- If the question is "what is the most common mammal in Nuova Guinea?", you must do a WEB search and may return "kangaroo".


## Create the Semantic Kernel Agent, based on Azure AI Foundry Agent

In [11]:
from semantic_kernel.agents import AgentRegistry

animalpicker_agent: AzureAIAgent = await AgentRegistry.create_from_yaml(
    yaml_str=yaml_str,
    client=project_client,
    settings=aiagent_settings,
)
animalpicker_agent

AzureAIAgent(arguments={'temperature': 0.0}, description="Agent that picks animals based on user's questions", id='asst_hZs0sAT1cH9jDlKDlmC9dVmb', instructions='. * YOUR GOAL **. - Return **JUST** an animal name.. . ** RULES **. - Run a WEB search with the provided tools.. - Do **NOT** use  your internal knowledge.. - Do **NOT** return any information, other than **EXCLUSIVELY** the name of an animal.. - Do **NOT** return any citations or sources.. . ** EXAMPLE **. - If the question is "what is the most common mammal in Nuova Guinea?", you must do a WEB search and may return "kangaroo".', kernel=Kernel(retry_mechanism=PassThroughWithoutRetry(), services={}, ai_service_selector=<semantic_kernel.services.ai_service_selector.AIServiceSelector object at 0x000001F27F899450>, plugins={}, function_invocation_filters=[], prompt_rendering_filters=[], auto_function_invocation_filters=[]), name='AnimalPicker', prompt_template=None, client=<azure.ai.projects.aio._patch.AIProjectClient object at 0x

## Invoke the agent

In [12]:
ANIMALPICKER_USER_INPUTS = [
    "What is the smallest reptile?", 
    "Which animal has the longest lifespan?",
]

await ChatWithAgentStreamAsync(animalpicker_agent, ANIMALPICKER_USER_INPUTS)


************************************
Message 0 from AuthorRole.USER: 'What is the smallest reptile?'

# AuthorRole.ASSISTANT: Brookesia nana【3:3†source】.

************************************
Message 0 from AuthorRole.USER: 'Which animal has the longest lifespan?'

# AuthorRole.ASSISTANT: Greenland Shark【3:4†source】.


# 3. Chat Completion Agent - `animaljoker_agent`

## Load the agent definition

In [13]:
# Read the agent template from the file
with open("./_agents/animal_joker.yaml", "r") as file:
    fstring_template = file.read()

# replace variables and fix carriage returns
animaljoker_agent_specs = eval(f"f'''{fstring_template}'''")
print(animaljoker_agent_specs)
yaml_str=f'{animaljoker_agent_specs[:animaljoker_agent_specs.find("instructions: ")]}{animaljoker_agent_specs[animaljoker_agent_specs.find("instructions: "):].replace('\n','. ')}'

type: chat_completion_agent
name: AnimalJoker
description: Agent that tells jokes about animals
model:
  id: gpt-4o
  options:
    temperature: 0.4
instructions: 
Given the input text, identify the animal mentioned in it.
Then, write exactly one joke or humorous story, about that animal. Joke must be:
- G rated.
- Workplace/family safe.
- Not longer than 20 words.

No sexism, racism or other bias/bigotry.

Be creative and funny. I want to laugh.

Your answer must start with 'Here is the joke - ' followed by the joke you invented.


## Create the Semantic Kernel Agent, based on AzureChatCompletion

In [14]:
from semantic_kernel.agents import AzureAIAgent, AgentRegistry

animaljoker_agent: AzureAIAgent = await AgentRegistry.create_from_yaml(
    yaml_str=yaml_str,
    kernel=kernel
)
animaljoker_agent

ChatCompletionAgent(arguments={'temperature': 0.4}, description='Agent that tells jokes about animals', id='912db3f7-443c-4162-9177-15428fa528ba', instructions=". Given the input text, identify the animal mentioned in it.. Then, write exactly one joke or humorous story, about that animal. Joke must be:. - G rated.. - Workplace/family safe.. - Not longer than 20 words.. . No sexism, racism or other bias/bigotry.. . Be creative and funny. I want to laugh.. . Your answer must start with 'Here is the joke - ' followed by the joke you invented.", kernel=Kernel(retry_mechanism=PassThroughWithoutRetry(), services={'chatcompletion_service_id': AzureChatCompletion(ai_model_id='gpt-4o', service_id='chatcompletion_service_id', instruction_role='system', client=<openai.lib.azure.AsyncAzureOpenAI object at 0x000001F27DDC74D0>, ai_model_type=<OpenAIModelTypes.CHAT: 'chat'>, prompt_tokens=0, completion_tokens=0, total_tokens=0)}, ai_service_selector=<semantic_kernel.services.ai_service_selector.AISer

## Invoke the agent

In [15]:
ANIMALJOKER_USER_INPUTS = [
    "Barbados threadsnake【3:0†source】.", 
    "Ostrich【3:1†source】.",
    "Greenland shark【3:0†source】.",
]

await ChatWithAgentStreamAsync(animaljoker_agent, ANIMALJOKER_USER_INPUTS)


************************************
Message 0 from AuthorRole.USER: 'Barbados threadsnake【3:0†source】.'

# AuthorRole.ASSISTANT: Here is the joke - Why didn't the threadsnake get invited to the party? Because it was too "knotty" for the dance floor!

************************************
Message 0 from AuthorRole.USER: 'Ostrich【3:1†source】.'

# AuthorRole.ASSISTANT: Here is the joke - Why did the ostrich bring a backpack to the zoo? It heard there were going to be lots of cheep trips!

************************************
Message 0 from AuthorRole.USER: 'Greenland shark【3:0†source】.'

# AuthorRole.ASSISTANT: Here is the joke - Why did the Greenland shark take a swim class? To finally stop sinking all its meetings!


# 4. [OpenAI Assistant Agent](https://learn.microsoft.com/en-us/semantic-kernel/frameworks/agent/agent-types/assistant-agent?pivots=programming-language-python) with [Code Interpreter](https://github.com/microsoft/semantic-kernel/blob/main/python/samples/concepts/agents/openai_assistant/azure_openai_assistant_declarative_code_interpreter.py) - `statistician_agent`

## Load the Agent definition

In [16]:
# Read the agent template from the file
with open("./_agents/statistician.yaml", "r") as file:
    fstring_template = file.read()

# replace variables and fix carriage returns
statistician_agent_specs = eval(f"f'''{fstring_template}'''")
print(statistician_agent_specs)
yaml_str=f'{statistician_agent_specs[:statistician_agent_specs.find("instructions: ")]}{statistician_agent_specs[statistician_agent_specs.find("instructions: "):].replace('\n','. ')}'

type: azure_assistant
name: Statistician
description: Agent that provides statistics about jokes
model:
  id: gpt-4o
  options:
    temperature: 0.0
tools:
  - type: code_interpreter
instructions: 
**YOUR GOAL**
Given the input text, identify the joke included in it.
Then analyze that joke to build a chart and calculate the "MAGIC NUMBER".

**MANDATORY RULES**
- **NEVER** try to interpret the meaning or do the semantic analysis of the given text. Consider it as a meaningless string.
- **NEVER** repeat the given joke, text or input.
- Do NOT ask **ANY** questions, just follow **ALL** the steps below **IN A SINGLE SHOT**.

**STEPS**
1) Extract the following three pieces of information from the text received:
A) Number of words.
B) Number of chars.
C) Number of spaces.

2) Create and save a bar chart showing the above three KPI's.

3) Calculate the MAGIC NUMBER as the sum of A, B and C

4) Produce a final sentence starting with "THE MAGIC NUMBER IS" followed by its value calculated in the

## Create the assistant client

In [17]:
from semantic_kernel.agents import AzureAssistantAgent
assistant_client = AzureAssistantAgent.create_client()
print(f"Assistant base URL: {assistant_client.base_url}")

Assistant base URL: https://mmoaiswc-01.openai.azure.com/openai/


## Create the assistant agent

In [18]:
from semantic_kernel.agents import AgentRegistry

statistician_agent: AzureAIAgent = await AgentRegistry.create_from_yaml(
    yaml_str=yaml_str,
    client=assistant_client
)
statistician_agent

AzureAssistantAgent(arguments={'temperature': 0.0}, description='Agent that provides statistics about jokes', id='asst_Poi7Olo77Ewb1M7dbbPctQeI', instructions='. **YOUR GOAL**. Given the input text, identify the joke included in it.. Then analyze that joke to build a chart and calculate the "MAGIC NUMBER".. . **MANDATORY RULES**. - **NEVER** try to interpret the meaning or do the semantic analysis of the given text. Consider it as a meaningless string.. - **NEVER** repeat the given joke, text or input.. - Do NOT ask **ANY** questions, just follow **ALL** the steps below **IN A SINGLE SHOT**.. . **STEPS**. 1) Extract the following three pieces of information from the text received:. A) Number of words.. B) Number of chars.. C) Number of spaces.. . 2) Create and save a bar chart showing the above three KPI\'s.. . 3) Calculate the MAGIC NUMBER as the sum of A, B and C. . 4) Produce a final sentence starting with "THE MAGIC NUMBER IS" followed by its value calculated in the previous step.'

## Invoke the agent

In [19]:
STATISTICIAN_USER_INPUTS = [
    "Why don't snakes use computers? Because they can't find the 'escape' key!",
]

await ChatWithAgentStreamAsync(statistician_agent, STATISTICIAN_USER_INPUTS)


************************************
Message 0 from AuthorRole.USER: 'Why don't snakes use computers? Because they can't find the 'escape' key!'

# AuthorRole.ASSISTANT: To proceed with the task, I will carry out the steps outlined:

1. Extract the three pieces of information from the text:
   A) Number of words.
   B) Number of characters.
   C) Number of spaces.

2. Create and save a bar chart for the above metrics.

3. Calculate the MAGIC NUMBER as the sum of A, B, and C.

4. Produce a final sentence with the MAGIC NUMBER.

Let's begin by extracting the required information:

```python
# Input text
text = "Why don't snakes use computers? Because they can't find the 'escape' key!"

# Calculating the required pieces of information
number_of_words = len(text.split())
number_of_chars = len(text)
number_of_spaces = text.count(' ')

number_of_words, number_of_chars, number_of_spaces
```

# AuthorRole.ASSISTANT: The extracted information is as follows:
- Number of words: 12
- Number of ch

# 5. [Responses Agent](https://learn.microsoft.com/en-us/semantic-kernel/frameworks/agent/agent-types/responses-agent?pivots=programming-language-python) with plugin and streaming - `reviewer_agent`
The OpenAI Responses API is OpenAI's most advanced interface for generating model responses. It supports text and image inputs, and text outputs. You are able to create stateful interactions with the model, using the output of previous responses as input. It is also possible to extend the model's capabilities with built-in tools for file search, web search, computer use, and more.

- [OpenAI Responses API](https://platform.openai.com/docs/api-reference/responses)
- [Responses API in Azure](https://learn.microsoft.com/en-us/azure/ai-foundry/openai/how-to/responses?tabs=python-secure)

## Load the agent definition

In [20]:
# Read the agent template from the file
with open("./_agents/reviewer.yaml", "r") as file:
    fstring_template = file.read()

# replace variables and fix carriage returns
reviewer_agent_specs = eval(f"f'''{fstring_template}'''")
print(reviewer_agent_specs)
yaml_str=f'{reviewer_agent_specs[:reviewer_agent_specs.find("instructions: ")]}{reviewer_agent_specs[reviewer_agent_specs.find("instructions: "):].replace('\n','. ')}'

type: openai_responses
name: Reviewer
description: Agent that reviews a sentence to check if it is satisfactory
model:
  id: gpt-4o
  options:
    temperature: 0.0
tools:
  - id: MagicNumberValidator.validate_magic_number
    type: function
instructions:
    Your goal is to review the text and say if we are satisfied or not with it, and explain the reason why you are / aren't satified.
    **FOLLOW THESE STEPS**
    1) Check if the text contains the 'MAGIC NUMBER'.
    2) If provided content does **NOT** contain the 'MAGIC NUMBER', please say that we are 'NOT SATISFIED'.
    3) If, instead, provided content **DOES CONTAIN** the magic number, please proceed as follows.
    3.A) If the magic number is successfully validated, it means that we are **SATISFIED**, please inform the user about this good news.
    3.B) If instead the magic number fails the validation process, then we are **NOT SATISFIED**, so please say that.
    3.C) Once the content has been updated in a subsequent response,

## Set up the client and model using Azure OpenAI Resources

In [21]:
from semantic_kernel.agents import AzureResponsesAgent

os.environ["AZURE_OPENAI_RESPONSES_DEPLOYMENT_NAME"] =  deployment_name
revieweragent_client = AzureResponsesAgent.create_client()

## Plugin

In [22]:
class MagicNumberValidator:
    from typing import Annotated
    from semantic_kernel.functions import kernel_function

    def __init__(self):
        self.validation = False

    @kernel_function(
        name="validate_magic_number",
        description="Validates the magic number",
    )
    def validate_mn(
        self,
        number: str,
    ) -> Annotated[str, "Validates the magic number"]:
        try:
            num = int(number)
            if (num % 2  == 0):
                self.validation = f"Validation was SUCCESSFUL for number {num}."
            else:
                self.validation = f"Validation FAILED for number {num}."
        except:
            self.validation = f"Validation FAILED for value {number}."
                
            
        return self.validation

mnv = MagicNumberValidator()
mnv.validate_mn("12")

'Validation was SUCCESSFUL for number 12.'

## Create the Semantic Kernel Agent, based on Azure OpenAI Responses Agent

In [23]:
from semantic_kernel.agents import AzureResponsesAgent

reviewer_agent = await AzureResponsesAgent.from_yaml(
    yaml_str=yaml_str,
    client=revieweragent_client,
    plugins=[MagicNumberValidator()],
)

# bug workaround
reviewer_agent.instructions = reviewer_agent.instruction_role
reviewer_agent.instruction_role = "developer"

reviewer_agent

AzureResponsesAgent(arguments={'temperature': 0.0}, description='Agent that reviews a sentence to check if it is satisfactory', id='8954d0fc-9395-488a-8c1f-ce289e32233a', instructions="Your goal is to review the text and say if we are satisfied or not with it, and explain the reason why you are / aren't satified. **FOLLOW THESE STEPS** 1) Check if the text contains the 'MAGIC NUMBER'. 2) If provided content does **NOT** contain the 'MAGIC NUMBER', please say that we are 'NOT SATISFIED'. 3) If, instead, provided content **DOES CONTAIN** the magic number, please proceed as follows. 3.A) If the magic number is successfully validated, it means that we are **SATISFIED**, please inform the user about this good news. 3.B) If instead the magic number fails the validation process, then we are **NOT SATISFIED**, so please say that. 3.C) Once the content has been updated in a subsequent response, please repeat al the steps 1-2-3.X.", kernel=Kernel(retry_mechanism=PassThroughWithoutRetry(), servic

## Invoke the Responses Agent

In [24]:
REVIEWER_USER_INPUTS = [
    "The magic number is 102", 
    "Today is a sunny day, and the magic nr is 103",
    "I will rain tomorrow",
]

await ChatWithAgentStreamAsync(reviewer_agent, REVIEWER_USER_INPUTS)


************************************
Message 0 from AuthorRole.USER: 'The magic number is 102'

# AuthorRole.ASSISTANT: We are **SATISFIED** because the magic number 102 was successfully validated. Great news!

************************************
Message 0 from AuthorRole.USER: 'Today is a sunny day, and the magic nr is 103'

# AuthorRole.ASSISTANT: We are **NOT SATISFIED** because the magic number "103" failed the validation process. Please provide the correct magic number to proceed.

************************************
Message 0 from AuthorRole.USER: 'I will rain tomorrow'

# AuthorRole.ASSISTANT: We are **NOT SATISFIED** because the provided text does not contain the 'MAGIC NUMBER'.


# Teardown

In [25]:
# delete all files
files_to_delete = await project_client.agents.files.list()
files_to_delete_nr = len(files_to_delete.data)

if files_to_delete_nr>0:
    i=0
    print(f"{files_to_delete_nr} files will now be deleted:")
    for f in files_to_delete.data:
        i += 1
        print(f"- File {i} of {files_to_delete_nr}: {f.filename} (id={f.id}) is being deleted...")
        await project_client.agents.files.delete(f.id)
else:
    print("No files to delete")

No files to delete


## Avoiding ***modifying a collection while iterating over it*** for both threads and agents

The code
```
threads_to_delete = project_client.agents.threads.list()
```
returns an async iterator that **lazily** fetches pages of threads.<br/>
But since we're deleting threads as we iterate, the underlying data source is being mutated during iteration. So when the iterator tries to fetch the next page, it hits a missing resource — hence the **ResourceNotFoundError**.<br/><br/>

This is a classic case of *modifying a collection while iterating over it*, which is risky even in synchronous code — and doubly so in async paged APIs.
### The solution
We need to fully materialize the list of threads before deleting anything. That way, the iterator isn’t affected by the deletions

In [26]:
# delete all threads

threads_to_delete = [t async for t in project_client.agents.threads.list()]
i = 0
for t in threads_to_delete:
    i += 1
    print(f"{i} - Thread <{t.id}> is being deleted...")
    await project_client.agents.threads.delete(thread_id=t.id)

1 - Thread <thread_uj2evZXc74JisVzUuPIl3zie> is being deleted...
2 - Thread <thread_Q7Rk70YJOp3OI42zEIQl6bHv> is being deleted...
3 - Thread <thread_upms6lMHwd6xQ7Eh79ENCpMT> is being deleted...
4 - Thread <thread_3g9H3G0Es7nnbUzP1p349DiG> is being deleted...
5 - Thread <thread_hBfmAu97s2ciElJn73OEtqF9> is being deleted...
6 - Thread <thread_0shBItMKW8iU2jejNz2CmGOs> is being deleted...
7 - Thread <thread_FcXKWthZIBUiFCrpKmhW2sUw> is being deleted...
8 - Thread <thread_Od7PVVYJgy8APXn7GpEXSNlK> is being deleted...
9 - Thread <thread_GAUSbIea9egDqhptFWDPeN1n> is being deleted...
10 - Thread <thread_zVxl6qSaP4QUSk6wTC91F6yY> is being deleted...
11 - Thread <thread_eWKHKkFlsU31s5ffBbMiDsMJ> is being deleted...
12 - Thread <thread_kxtGEL5Ui1iQKsyrchGUk72k> is being deleted...
13 - Thread <thread_wut8Yf8FsireF0n7t275dAlz> is being deleted...
14 - Thread <thread_Cp1pUcAgq0P8EN0lxYnkWAPY> is being deleted...
15 - Thread <thread_1ftoHdKUJdPOXxuKNV0TPnz3> is being deleted...
16 - Thread <thread

In [27]:
# delete all agents

agents_to_delete = [a async for a in project_client.agents.list_agents(limit=100)]
i=0
for a in agents_to_delete:
    i += 1
    print(f"{i} - Agent <{a.id}> is being deleted...")
    await project_client.agents.delete_agent(agent_id=a.id)

1 - Agent <asst_hZs0sAT1cH9jDlKDlmC9dVmb> is being deleted...
2 - Agent <asst_wzbkq6ShiZ10pWgaqmSdX4D8> is being deleted...
3 - Agent <asst_kI14ZkgJXyyIg67s2UDQhAYY> is being deleted...
